In [ ]:
import json

In [105]:
split = 'test'
# split = 'train'
# split = 'val'

mode = 'cev' 
# mode = 'cvi' 

inf = True

In [107]:
import json
def read_json_object(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        json_data = json.load(file)
    return json_data

In [108]:
train = read_json_object(f"./LIAR-RAW/{split}.json")

In [109]:
def clean_wi_inf(raw_list, mode):
    conv_list = []
    for event in raw_list:
        conv = {}
        conv['id'] = event['event_id']
        
        claim = event['claim']
        label = event['label']
        inf = event['explain']
        if label in {'pants-fire', 'false', 'barely-true'}:
            new_label = 'FALSE'
        elif label in {'mostly-true', 'true'}:
            new_label = 'TRUE'
        else:
            new_label = 'HALF-TRUE'

        evidences = ''
        reports_list = event['reports']
        for rep_dict in reports_list:
            tokenized_list = rep_dict['tokenized']
            for rep in tokenized_list:
                if rep["is_evidence"] != 0:                   
                    evidences += rep['sent']

        if len(evidences) == 0:
            continue
        
        utter_list = []
        if mode == 'cev':
            utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
        else:
            utter_list.append({"from": "human", "value": f"Claim: {claim}"})
        utter_list.append({"from": "gpt", "value": f"Verdict: {new_label}. Explanation: {inf}"})

        conv['conversations'] = utter_list
        conv_list.append(conv)
    return conv_list
   

In [110]:
def clean(raw_list, mode):
    conv_list = []
    for event in raw_list:
        conv = {}
        conv['id'] = event['event_id']
        
        claim = event['claim']
        label = event['label']
        if label in {'pants-fire', 'false', 'barely-true'}:
            new_label = 'FALSE'
        elif label in {'mostly-true', 'true'}:
            new_label = 'TRUE'
        else:
            new_label = 'HALF-TRUE'

        evidences = ''
        evidence_id = 1 # 
        reports_list = event['reports']
        for rep_dict in reports_list:
            tokenized_list = rep_dict['tokenized']
            for rep in tokenized_list:
                if rep["is_evidence"] != 0:                  
                    evidences += (f"({evidence_id}) {rep['sent']}") 
                    evidence_id += 1

        if len(evidences) == 0:
            continue
        
        utter_list = []
        if mode == 'cev':
            utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
        else:
            utter_list.append({"from": "human", "value": f"Claim: {claim}"})
        utter_list.append({"from": "gpt", "value": f"Verdict: {new_label}"})
        conv['conversations'] = utter_list
        conv_list.append(conv)
    return conv_list
   

In [111]:
if inf:
    data = clean_wi_inf(train, mode)
else:
    data = clean(train, mode)
len(data)

855

In [112]:
data[-1]

{'id': '6743.json',
 'conversations': [{'from': 'human',
   'value': 'Claim: Says the governor is going around the state talking about [how] we should fund an income tax cut that benefits higher income earners and not lower income earners'},
  {'from': 'gpt',
   'value': 'Verdict: FALSE. Explanation: Burzichelli said, "the governor is going around the state talking about [how] we should fund an income tax cut" that benefits higher income earners and not lower income earners. Christie originally proposed an across-the-board income tax cut that would have benefited wealthy individuals more than those with lower-incomes, but he has since backed off that plan. His new proposal is based on the property tax burden of homeowners and excludes residents with incomes more than $400,000. To comment on this ruling, go to NJ.com.'}]}

In [113]:
data[0]

{'id': '11972.json',
 'conversations': [{'from': 'human',
   'value': 'Claim: Building a wall on the U.S.-Mexico border will take literally years.'},
  {'from': 'gpt',
   'value': 'Verdict: TRUE. Explanation: Perry said: "Building a wall" on the U.S.-Mexico border "will take literally years." If Trump has a fast-track plan to plan the wall, purchase required land, complete needed studies and erect the wall in a year or less, it’s not public. Meantime, engineering experts agree the wall would most likely take years to complete. Keep in mind, too, it took more than six years to build roughly 700 miles of fence and barriers along the roughly 2,000-mile U.S.-Mexico border. Click here formore on the six PolitiFact ratings and how we select facts to check.'}]}

In [116]:
with open(f'./liar-raw-post/cvi/{split}_liar_raw_{mode}.json', 'w', encoding="utf-8" ) as file:
    json.dump(data, file, indent=2, ensure_ascii=False)